# 시퀀스 모델 복습

이 노트북은 `04_sequence_model`의 RNN, LSTM, GRU, NSMC 감성 분류 흐름을 문제로 복습합니다. 총 26문제이며, 각 문제 아래의 빈 코드 셀에 풀이를 작성하세요.

- 문제는 순서 정보와 RNN의 shape에서 시작해 게이트 구조, 패딩 처리, 실제 한국어 리뷰 분류로 진행됩니다.
- 값 하나를 외우기보다 입력·출력 shape, 실제 길이, 로짓과 확률의 차이를 함께 확인하세요.
- 작은 예제의 성능은 일반화 성능이 아닙니다. 점수가 나온 이유와 한계를 주석으로 설명해 보세요.

## 원본 노트북과 문제 연결

- `01_sequential_data_rnn.ipynb`: 문제 1~8
- `02_lstm.ipynb`: 문제 9~14
- `03_gru.ipynb`: 문제 15~20
- `04_nsmc.ipynb`: 문제 21~26

각 문제는 이전 문제의 변수나 개념을 일부 사용할 수 있습니다. 셀을 위에서부터 순서대로 푸는 것을 권장합니다.

In [ ]:
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. 순차 데이터와 기본 RNN

이 절에서는 입력 순서가 달라질 때 표현이 달라지는 이유와, 기본 RNN이 마지막 은닉 상태로 시퀀스 하나를 분류하는 흐름을 확인합니다.

## 문제 1. 같은 값, 다른 순서

다음 두 시퀀스를 만들고 비교하세요.

    ascending = torch.tensor([1.0, 2.0, 3.0, 4.0])
    descending = torch.tensor([4.0, 3.0, 2.0, 1.0])

조건:

- 두 시퀀스의 합과 위치별 값이 모두 같은지 출력하세요.
- 두 시퀀스를 Count/BoW 방식으로 단순히 원소 빈도만 세면 왜 구분하기 어려운지 주석으로 설명하세요.
- 자연어에서 순서가 달라져 의미가 바뀌는 짧은 예시를 하나 작성하세요.

## 문제 2. RNN 입력과 반환값의 shape

`batch_first=True`인 RNN에 아래 크기의 입력을 전달하세요.

    B, L, F, H = 2, 4, 3, 5
    rnn_input = torch.randn(B, L, F)

조건:

- `nn.RNN(input_size=F, hidden_size=H, batch_first=True)`를 만들고 `output, hidden`을 받으세요.
- 입력, output, hidden의 shape를 출력하세요.
- 각 축의 B, L, F, H가 무엇인지 주석으로 작성하세요.
- 단층·단방향 RNN에서 `output[:, -1, :]`와 `hidden[-1]`이 같은지 `torch.allclose`로 확인하세요.

## 문제 3. 마지막 은닉 상태를 로짓으로 바꾸기

문제 2의 마지막 은닉 상태를 입력으로 이진 분류 로짓을 만드세요.

조건:

- `nn.Linear(H, 1)`을 만들고 `hidden[-1]`에 적용하세요.
- 로짓의 shape와 `torch.sigmoid(logits)`의 shape를 출력하세요.
- 로짓과 확률이 각각 무엇인지 설명하세요.
- 확률이 0.5 이상일 때 클래스 1로 판단하는 예측 텐서를 만드세요.

## 문제 4. 순서 분류 데이터 만들기

길이 12의 상승·하강 수열을 각각 40개씩 만들고, 작은 정규분포 잡음을 추가하세요.

조건:

- 상승 수열의 레이블은 1, 하강 수열의 레이블은 0으로 두세요.
- 최종 입력 shape를 `(80, 12, 1)`, 레이블 shape를 `(80,)`로 만드세요.
- 클래스별 개수를 출력하세요.
- 학습/검증을 64개/16개로 나누되 두 클래스가 모두 들어가도록 구성하세요.

## 문제 5. RNN 분류기 클래스 구현

입력 특성 1개를 받아 상승·하강을 분류하는 `SequenceRNNClassifier`를 구현하세요.

조건:

- `__init__`에 `nn.RNN`과 `nn.Linear`를 정의하세요.
- `forward`에서 RNN의 마지막 은닉 상태를 꺼내 분류 계층에 전달하세요.
- `hidden_size=8` 모델을 만들고 문제 4의 입력 일부를 전달하세요.
- 출력은 확률이 아닌 `(배치,)` 모양의 로짓이 되게 하세요.

## 문제 6. BCEWithLogitsLoss로 한 배치 학습

문제 5의 모델과 문제 4의 학습 데이터를 사용해 한 번의 학습 단계를 수행하세요.

조건:

- `BCEWithLogitsLoss`와 Adam optimizer를 사용하세요.
- 레이블 dtype과 로짓 shape가 손실 함수의 요구사항에 맞도록 변환하세요.
- `zero_grad() → forward → loss → backward() → step()` 순서로 구현하세요.
- 손실 계산 전에 sigmoid를 적용하면 안 되는 이유를 주석으로 설명하세요.

## 문제 7. 학습·검증 루프와 정확도

문제 5 모델을 30 epoch 이하로 학습하고 검증 정확도를 구하세요.

조건:

- 학습 데이터는 `DataLoader(..., shuffle=True)`로 만드세요.
- 각 epoch의 평균 학습 손실을 기록하세요.
- 검증 시 `model.eval()`과 `torch.no_grad()`를 사용하세요.
- 검증 로짓을 확률과 0/1 예측으로 변환한 뒤 accuracy를 출력하세요.

## 문제 8. 기본 RNN의 한계 점검

기본 RNN의 장기 의존성 문제를 글과 코드로 정리하세요.

조건:

- 시퀀스 길이 12와 100에서 역전파가 어려워질 수 있는 이유를 설명하세요.
- 기울기 소실과 기울기 폭주를 각각 한 문장으로 설명하세요.
- `torch.nn.utils.clip_grad_norm_`을 학습 루프에 추가하는 코드를 작성하세요.
- gradient clipping이 기울기 소실까지 해결하지는 못하는 이유를 주석으로 남기세요.

# 2. LSTM의 셀 상태와 게이트

LSTM은 셀 상태와 게이트로 기억을 선택적으로 유지합니다. 이 절에서는 LSTM의 반환값, 게이트 순서, 양방향 구조를 확인합니다.

## 문제 9. LSTM의 output·hidden·cell shape

문제 2의 `rnn_input`을 같은 크기의 단층·단방향 LSTM에 전달하세요.

조건:

- `nn.LSTM(F, H, batch_first=True)`를 만들고 `output, (hidden, cell)`을 받으세요.
- 세 반환값의 shape를 출력하세요.
- hidden과 cell의 역할 차이를 설명하세요.
- `output[:, -1, :]`와 `hidden[-1]`이 같은지 확인하세요.

## 문제 10. LSTM 게이트 가중치 분리

문제 9의 LSTM에서 `weight_ih_l0`, `weight_hh_l0`, 두 bias 텐서를 확인하고 게이트별로 나누세요.

조건:

- 각 원본 텐서의 shape를 출력하세요.
- `torch.chunk(..., 4, dim=0)`으로 입력·망각·후보·출력 게이트를 분리하세요.
- PyTorch LSTM의 저장 순서 `i, f, g, o`를 주석으로 표시하세요.
- 각 분리된 입력 가중치의 shape를 출력하세요.

## 문제 11. 한 타임 스텝의 LSTM 계산 검증

문제 9 LSTM의 첫 번째 입력 위치와 0으로 초기화한 상태를 사용해 게이트 계산을 직접 구현하세요.

조건:

- i, f, o에는 sigmoid, g에는 tanh를 적용하세요.
- `cell_t = f * cell_prev + i * g`, `hidden_t = o * tanh(cell_t)`로 계산하세요.
- 직접 계산한 hidden/cell과 LSTM이 반환한 첫 위치의 값을 `allclose`로 비교하세요.
- `*`가 행렬곱이 아닌 원소별 곱이라는 점을 주석으로 작성하세요.

## 문제 12. RNN과 LSTM의 파라미터 수 비교

같은 `input_size=3`, `hidden_size=5`로 RNN과 LSTM을 만들고 학습 가능 파라미터 수를 비교하세요.

조건:

- `sum(p.numel() for p in model.parameters())`로 파라미터 수를 세세요.
- 입력·순환 가중치와 bias의 shape를 모두 출력하세요.
- LSTM의 수가 더 큰 이유를 게이트 수와 연결해 설명하세요.
- 파라미터 수가 많다고 항상 더 좋은 모델은 아닌 이유를 한 문장으로 작성하세요.

## 문제 13. 다층·양방향 LSTM의 최종 문장 벡터

`num_layers=2`, `bidirectional=True`인 LSTM을 만들고 문제 2 입력을 전달하세요.

조건:

- output, hidden, cell의 shape를 출력하세요.
- hidden에서 마지막 층의 정방향과 역방향 상태를 꺼내 특성 축으로 연결하세요.
- 문장 벡터 shape가 `(B, 2 * H)`인지 확인하세요.
- `hidden[-1]` 하나만 쓰면 어느 방향의 정보를 잃는지 설명하세요.

## 문제 14. LSTM 분류기의 출력층

문제 13에서 만든 양방향 문장 벡터를 이진 분류 로짓으로 바꾸세요.

조건:

- `nn.Linear(2 * H, 1)`을 사용하세요.
- dropout을 문장 벡터와 분류 계층 사이에 넣는 예시를 작성하세요.
- 학습 모드와 평가 모드에서 dropout이 어떻게 다르게 동작하는지 설명하세요.
- 레이블이 3개인 다중 분류라면 출력층과 손실 함수를 어떻게 바꿀지 작성하세요.

# 3. GRU와 가변 길이 문장

GRU는 셀 상태를 별도로 두지 않고 업데이트·리셋 게이트로 은닉 상태를 갱신합니다. 이어서 패딩을 제외해 가변 길이 문장을 처리하는 방법을 연습합니다.

## 문제 15. GRU 반환값과 LSTM 차이

문제 2의 입력을 GRU에 전달하고 LSTM의 반환값과 비교하세요.

조건:

- `nn.GRU(F, H, batch_first=True)`의 output과 hidden shape를 출력하세요.
- LSTM과 달리 cell이 없는 이유를 설명하세요.
- 단층·단방향 GRU에서 `output[:, -1, :]`와 `hidden[-1]`을 비교하세요.
- GRU와 LSTM의 공통 목적을 한 문장으로 정리하세요.

## 문제 16. GRU 게이트 직접 계산

문제 15 GRU의 첫 타임 스텝을 직접 계산하세요.

조건:

- `weight_ih_l0`과 `weight_hh_l0`을 리셋 r, 업데이트 z, 후보 n으로 분리하세요.
- PyTorch의 게이트 저장 순서가 `r, z, n`임을 주석으로 작성하세요.
- r, z, n과 현재 hidden을 계산하고 모델의 첫 위치 output과 비교하세요.
- 업데이트 게이트가 이전 상태와 후보 상태를 섞는 역할을 설명하세요.

## 문제 17. RNN·GRU·LSTM 파라미터 수 표

같은 F=3, H=5 조건에서 세 순환 계층의 파라미터 수를 표 형태로 출력하세요.

조건:

- RNN, GRU, LSTM을 각각 생성하세요.
- 모델명, 파라미터 수, 게이트/상태 특징을 담은 리스트 또는 DataFrame을 만드세요.
- 파라미터 수의 크기 순서를 확인하세요.
- 데이터가 적고 학습 시간이 제한될 때 GRU를 고려할 수 있는 이유를 작성하세요.

## 문제 18. PAD·OOV를 포함한 토큰 ID 만들기

다음 학습 문장만 사용해 단어 사전을 만드세요.

    train_sentences = ['영화 정말 재미 있다', '영화 전혀 재미 없다', '배우 연기 좋다']
    new_sentences = ['영화 정말 좋다', '처음 보는 단어']

조건:

- 공백 분리 토큰화 후 Counter로 빈도를 세세요.
- PAD는 0, OOV는 1, 나머지 단어는 2부터 ID를 부여하세요.
- 새 문장을 사전으로 정수 인코딩하고 미등록 단어가 OOV가 되는지 확인하세요.
- 단어 ID의 숫자 크기는 의미나 중요도 순서가 아니라는 점을 주석으로 작성하세요.

## 문제 19. 오른쪽 패딩과 실제 길이

문제 18의 인코딩 결과를 최대 길이 6으로 맞추세요.

조건:

- 길이가 6보다 길면 뒤를 자르고, 짧으면 오른쪽에 PAD를 붙이세요.
- 입력 dtype을 `torch.long`, shape를 `(문장 수, 6)`으로 만드세요.
- PAD를 제외한 실제 길이를 별도 long tensor로 만드세요.
- 빈 문자열이 들어온다면 OOV 하나로 바꿔 최소 길이 1을 지키는 코드를 작성하세요.

## 문제 20. Embedding + packed GRU 분류기

문제 19의 토큰 ID와 길이를 처리하는 `GRUSentenceClassifier`를 구현하세요.

조건:

- Embedding은 `padding_idx=PAD_ID`로 설정하세요.
- `pack_padded_sequence(..., batch_first=True, enforce_sorted=False)`를 사용하세요.
- GRU의 마지막 hidden을 `nn.Linear`에 전달해 문장별 로짓 하나를 반환하세요.
- 입력 ID, embedding 출력, 로짓의 shape를 주석으로 표시하세요.

# 4. NSMC 한국어 영화 리뷰 감성 분류

마지막 절에서는 제공된 NSMC 파일을 읽고, train 전용 어휘와 BiGRU 모델을 연결한 뒤 평가 결과를 해석합니다. 전체 데이터를 모두 학습할 필요 없이 작은 표본으로 흐름을 확인해도 됩니다.

## 문제 21. NSMC 파일 읽기와 결측치 확인

`04_sequence_model/data/nsmc/ratings_train.txt`를 읽어 DataFrame을 만드세요.

조건:

- 노트북 위치를 기준으로 상대 경로를 구성하세요.
- `id`, `document`, `label` 열, 행 수, 결측치 수를 출력하세요.
- document 결측치를 제거한 뒤 label별 개수를 확인하세요.
- train/validation/test가 서로 다른 역할을 가져야 하는 이유를 설명하세요.

## 문제 22. 한국어 정제 함수 구현

문자열에서 한글과 공백만 남기고 연속 공백을 하나로 정리하는 `clean_korean` 함수를 작성하세요.

조건:

- 정규표현식을 사용하세요.
- `'영화 정말 좋았어요!!! 10점 :)'`과 숫자·기호만 있는 문자열을 각각 테스트하세요.
- 빈 결과가 될 수 있는 이유를 설명하세요.
- 형태소 분석기를 사용할 수 있다면 train·validation·test에 같은 토큰화 함수를 적용해야 하는 이유를 작성하세요.

## 문제 23. train 전용 어휘 만들기

작은 train 표본의 정제된 문장을 공백 분리해 토큰화하고, 빈도 상위 5,000개까지 사전을 만드세요.

조건:

- PAD=0, OOV=1을 먼저 등록하세요.
- validation 또는 test 문장으로 사전을 fit하지 마세요.
- 사전 크기와 빈도 상위 토큰 10개를 출력하세요.
- 평가 데이터로 어휘를 학습하면 왜 데이터 누수인지 설명하세요.

## 문제 24. NSMC용 BiGRU 모델 구현

토큰 ID와 실제 길이를 받아 긍정 로짓을 반환하는 `NSMCBiGRU`를 구현하세요.

조건:

- Embedding → packed sequence → 한 층 양방향 GRU → dropout → Linear 순서로 구성하세요.
- 양방향 마지막 hidden 두 개를 연결한 뒤 분류하세요.
- `embedding_dim=64`, `hidden_size=64`일 때 분류 계층의 입력 차원을 명시하세요.
- 정렬되지 않은 배치도 처리하도록 `enforce_sorted=False`를 설정하세요.

## 문제 25. 학습·검증 분리와 오분류 수집

작은 학습/검증 표본으로 모델을 학습하고 오분류 사례를 모으세요.

조건:

- train loader만 `shuffle=True`로 설정하세요.
- validation에서는 `eval()`과 `no_grad()`를 사용하세요.
- 문장, 정답, 예측, 긍정 확률을 포함하는 오분류 DataFrame을 만드세요.
- 확률이 0.5 부근인 오분류와 0 또는 1 부근의 오분류가 각각 시사하는 점을 작성하세요.

## 문제 26. 평가 지표와 개선 실험 설계

모델의 예측과 정답으로 정확도, confusion matrix, classification report를 출력하고 다음 실험을 설계하세요.

조건:

- confusion matrix의 행과 열이 각각 무엇을 뜻하는지 주석으로 작성하세요.
- precision, recall, F1 중 감성 분류에서 특히 확인할 지표와 이유를 작성하세요.
- 어휘 크기, 최대 길이, hidden 크기, epoch 중 한 번에 하나만 바꾸는 실험 계획을 2개 제시하세요.
- test 성능을 보고 난 뒤 test 결과에 맞춰 반복 튜닝하면 안 되는 이유를 설명하세요.